In [6]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import json
from pathlib import Path

api_dir = Path("../data/image_api_calls")

# Getting the unsplash search JSON sorted according to number
# api_files = sorted(api_dir.glob("unsplash_search_*.json"), key=lambda path: int(path.stem.rsplit("_", 1)[-1]))
# Cause I have test file as well so above will give error

api_files = sorted(
    [path for path in api_dir.glob("unsplash_search_*.json") if path.stem.rsplit("_", 1)[-1].isdigit()],
    key=lambda path: int(path.stem.rsplit("_", 1)[-1])
)

print("Number of API response files:", len(api_files))

Number of API response files: 9


In [3]:
# Extract Image Metadata
all_retrieved_images = []

for file_path in api_files:

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    query_number = int(file_path.stem.rsplit("_", 1)[-1])

    for photo in data["results"]:

        image_record = {
            "image_id": photo["id"],
            "image_url": photo["urls"]["small"],
            "source_url": photo["links"]["html"],
            "description": (
                photo.get("description")
                or photo.get("alt_description")
            ),
            "photographer": photo["user"]["name"],
            "photographer_url": photo["user"]["links"]["html"],
            "query_number": query_number
        }

        all_retrieved_images.append(image_record)

print("Total retrieved records:", len(all_retrieved_images))

Total retrieved records: 43


In [4]:
# Deduplicate Retreived Images
unique_images = {}

for image in all_retrieved_images:

    image_id = image["image_id"]

    if image_id not in unique_images:
        unique_images[image_id] = image

retrieved_images = list(unique_images.values())

print("Total records:", len(all_retrieved_images))
print("Unique images:", len(retrieved_images))
print("Duplicates removed:", (len(all_retrieved_images) - len(retrieved_images)))

Total records: 43
Unique images: 19
Duplicates removed: 24


In [7]:
# Regenerate Context Embedding

import torch
from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = "patrickjohncyh/fashion-clip"

model = CLIPModel.from_pretrained(MODEL_NAME)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

model.eval()

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 517.49it/s]


CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1

In [8]:
with open("../data/context.json", "r", encoding="utf-8") as file:
    context = json.load(file)

In [9]:
print(context)

{'destination': 'Seoul', 'season': 'winter', 'weather_clothing_requirements': {'outerwear': ['winter coat', 'puffer jacket'], 'top': ['sweater', 'knitwear'], 'bottom': ['long trousers'], 'footwear': ['closed shoes', 'boots'], 'properties': ['warm', 'insulated', 'layerable', 'weather-resistant']}, 'activity_style_requirements': {'sightseeing': ['casual', 'comfortable'], 'dinner': ['smart casual', 'polished']}, 'user_style': ['minimalist']}


In [ ]:
# Creating the Expanded Fashion Context
import sys
if ".." not in sys.path:
    sys.path.insert(0, "..")

from src.context_builder import build_expanded_fashion_context
expanded_fashion_context=build_expanded_fashion_context(context)

print(expanded_fashion_context)

Seoul winter winter coat puffer jacket sweater knitwear long trousers closed shoes boots warm insulated layerable weather-resistant sightseeing casual comfortable dinner smart casual polished minimalist


In [14]:
# Tokenize the ExpandedFashionContext
inputs = processor(
    text=[expanded_fashion_context],
    return_tensors="pt"
)

print(inputs)

{'input_ids': tensor([[49406, 13253,  2541,  2541,  7356,   755, 11636,  6164, 11455, 18745,
          2839,  1538, 23172,  4552,  4079,  7319,  3616, 42051,  7205, 18801,
          2237,   268, 17542, 34210, 10129,  8652,  2571,  4115, 10129, 21478,
         26641, 49407]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1]])}


In [15]:
# Inspect the tokens
input_ids = inputs["input_ids"]

print("Input IDs:")
print(input_ids)

print("\nShape:")
print(input_ids.shape)

Input IDs:
tensor([[49406, 13253,  2541,  2541,  7356,   755, 11636,  6164, 11455, 18745,
          2839,  1538, 23172,  4552,  4079,  7319,  3616, 42051,  7205, 18801,
          2237,   268, 17542, 34210, 10129,  8652,  2571,  4115, 10129, 21478,
         26641, 49407]])

Shape:
torch.Size([1, 32])


In [18]:
print("Maximum token count allowed in FashionCLIP:", model.config.text_config.max_position_embeddings)

Maximum token count allowed in FashionCLIP: 77


In [19]:
# torch.no_grad() : Don't track gradients because we're not performing backpropagation but still maintains some internal tensor bookkeeping, such as version counters.
# Hence, we will use torch.inference_mode() : This also disables gradient tracking, but skips additional overhead such as view tracking and version-counter updates.
with torch.inference_mode():
    text_output = model.get_text_features(**inputs)

context_embedding = text_output.pooler_output

print(context_embedding.shape)

torch.Size([1, 512])


In [20]:
# Normalize the Embedding
print("Norm of Context Embedding Before Normalization :", context_embedding.norm(dim=-1))

Norm of Context Embedding Before Normalization : tensor([7.3051])


In [ ]:
context_embedding = context_embedding / context_embedding.norm(dim=-1, keepdim=True)
# Pytorch uses broadcasting so the norm is divided to each embedding vector value
# Can also use below for normalization
# context_embedding = torch.nn.functional.normalize(context_embedding, p=2, dim=-1)
print("Norm of Context Embedding After Normalization:", context_embedding.norm(dim=-1))

Norm of Context Embedding After Normalization: tensor([1.])


In [22]:
import requests
from PIL import Image
from io import BytesIO

def load_image_from_url(image_url):
    response = requests.get(image_url, timeout=20)
    response.raise_for_status()

    image = Image.open(BytesIO(response.content)).convert("RGB")

    return image

In [23]:
images = []
valid_image_records = []

for image_record in retrieved_images:
    try:
        image = load_image_from_url(image_record["image_url"])

        images.append(image)
        valid_image_records.append(image_record)

    except Exception as e:
        print(f"Failed to load {image_record['image_id']}: {e}")

print("Images successfully loaded:", len(images))

Images successfully loaded: 19


In [ ]:
# converts your 19 RGB images into the numerical tensors expected by FashionCLIP's Vision Transformer.
image_inputs = processor(
    images=images,
    return_tensors="pt"
)

In [25]:
print(image_inputs.keys())
print(image_inputs["pixel_values"].shape)

KeysView({'pixel_values': tensor([[[[ 0.7917,  0.8647,  0.5581,  ..., -0.9456, -0.6536, -0.5806],
          [ 0.8209,  0.9960,  0.8938,  ..., -0.9456, -0.6098, -0.5368],
          [ 1.0398,  1.1712,  1.2150,  ..., -0.9602, -0.6390, -0.5660],
          ...,
          [-0.8726, -1.1207, -1.1353,  ..., -1.0623, -0.8580, -0.3908],
          [-0.6682, -1.0039, -1.1791,  ..., -1.1937, -1.0915, -0.7704],
          [-0.8142, -1.2083, -1.3105,  ..., -1.1353, -1.0477, -0.8434]],

         [[ 0.7542,  0.8593,  0.5141,  ..., -1.1518, -0.8666, -0.8366],
          [ 0.8292,  1.0093,  0.8743,  ..., -1.1368, -0.8366, -0.7766],
          [ 1.0994,  1.2344,  1.2495,  ..., -1.1518, -0.8516, -0.7916],
          ...,
          [-1.0918, -1.3319, -1.3319,  ..., -1.3169, -1.2118, -0.8516],
          [-0.8967, -1.2118, -1.3769,  ..., -1.4069, -1.4369, -1.2118],
          [-1.0617, -1.4219, -1.5270,  ..., -1.3019, -1.3319, -1.2568]],

         [[ 0.8803,  0.8945,  0.4821,  ..., -0.8972, -0.6412, -0.6128],
    

In [26]:
# Generate the 17 FashionCLIP image embeddings
with torch.inference_mode():
    image_output = model.get_image_features(**image_inputs)

image_embeddings = image_output.pooler_output

print("Image embeddings shape:", image_embeddings.shape)

Image embeddings shape: torch.Size([19, 512])


In [27]:
# L2-normalize each image independently

image_embeddings = image_embeddings / image_embeddings.norm(dim=-1, keepdim=True)

In [28]:
print("Image embeddings shape:", image_embeddings.shape)
print("Image embedding norms:", image_embeddings.norm(dim=-1))

Image embeddings shape: torch.Size([19, 512])
Image embedding norms: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000])


In [29]:
# Cosine Similarity between the context embedding and image embeddings (Embedded via FashionClip)
similarities = context_embedding @ image_embeddings.T
print("Similarity shape:", similarities.shape)

Similarity shape: torch.Size([1, 19])


In [31]:
# Rank the Images
scores = similarities.squeeze(0)

ranked_indices = torch.argsort(scores,descending=True)

# printing the context again
print(expanded_fashion_context)
print()
for rank, idx_tensor in enumerate(ranked_indices, start=1):
    image_index = idx_tensor.item()
    # print(rank, image_paths[image_index].name, scores[image_index].item())
    print(f"{rank}. Image ID: {valid_image_records[image_index]['image_id']} | Similarity: {scores[image_index].item():.4f}")


Seoul winter winter coat puffer jacket sweater knitwear long trousers closed shoes boots warm insulated layerable weather-resistant sightseeing casual comfortable dinner smart casual polished minimalist

1. Image ID: sxZeyJHpKeM | Similarity: 0.2959
2. Image ID: eVtwC7k7BR4 | Similarity: 0.2769
3. Image ID: gb53aVsnQbg | Similarity: 0.2486
4. Image ID: ixLo8pqKC-o | Similarity: 0.2467
5. Image ID: 2A4CxA33aA8 | Similarity: 0.2380
6. Image ID: AStrLSqUilU | Similarity: 0.2371
7. Image ID: pKjBsW0dsmo | Similarity: 0.2229
8. Image ID: vIgBI6iNrkE | Similarity: 0.2229
9. Image ID: W6obGIZGai0 | Similarity: 0.2183
10. Image ID: _nws4tHTVto | Similarity: 0.2154
11. Image ID: cdlrjH_8g48 | Similarity: 0.2115
12. Image ID: xbMioPyoa7E | Similarity: 0.2042
13. Image ID: Dr734ID7WiI | Similarity: 0.1984
14. Image ID: rXANayuznKI | Similarity: 0.1896
15. Image ID: HdKYNcBHjYA | Similarity: 0.1852
16. Image ID: Muxr4P0OVdE | Similarity: 0.1727
17. Image ID: H9-y15Q88MY | Similarity: 0.1651
18. Im

In [32]:
TOP_N = 10
top_indices = ranked_indices[:TOP_N]
print("Number of selected images:", len(top_indices))

Number of selected images: 10


In [33]:
from IPython.display import display, HTML
from html import escape

for rank, idx in enumerate(top_indices, start=1):

    image_index = idx.item()

    image = valid_image_records[image_index]

    image_url = escape(image["image_url"], quote=True)
    source_url = escape(image["source_url"], quote=True)
    photographer = escape(image["photographer"])
    photographer_url = escape(image["photographer_url"], quote=True)

    similarity = scores[image_index].item()

    display(HTML(f"""
        <div style="margin-bottom:25px;">

            <h4>Rank {rank} | Similarity: {similarity:.4f}</h4>

            <img src="{image_url}"
                 style="width:300px;
                        height:400px;
                        object-fit:contain;">

            <p>
                Photo by
                <a href="{photographer_url}" target="_blank">
                    {photographer}
                </a>
                on
                <a href="{source_url}" target="_blank">
                    Unsplash
                </a>
            </p>

        </div>
    """))

In [34]:
# Output Preperation for Handover to Next Stage

top_indices = ranked_indices[:TOP_N]

top_image_embeddings = image_embeddings[top_indices]

print("Top image embeddings shape:", top_image_embeddings.shape)

Top image embeddings shape: torch.Size([10, 512])


In [35]:
top_images = []

for rank, idx in enumerate(top_indices, start=1):

    image_index = idx.item()

    image_record = valid_image_records[image_index].copy()

    image_record["rank"] = rank
    image_record["similarity_score"] = scores[image_index].item()

    top_images.append(image_record)

In [36]:
phase_2_output = {
    "context_embedding": context_embedding,
    "top_images": top_images,
    "top_image_embeddings": top_image_embeddings
}

In [37]:
# Saving the output
from pathlib import Path
import torch

output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "phase_2_output.pt"

torch.save(phase_2_output, output_path)

print("Saved to:", output_path)

Saved to: ..\outputs\phase_2_output.pt


In [38]:
# For Loading

loaded_output = torch.load(
    output_path,
    map_location="cpu",
    weights_only=True
)

print("Context embedding shape:",
      loaded_output["context_embedding"].shape)

print("Number of top images:",
      len(loaded_output["top_images"]))

print("Top image embeddings shape:",
      loaded_output["top_image_embeddings"].shape)

Context embedding shape: torch.Size([1, 512])
Number of top images: 10
Top image embeddings shape: torch.Size([10, 512])


In [40]:
print(
    "Context embeddings identical:",
    torch.equal(
        phase_2_output["context_embedding"].cpu(),
        loaded_output["context_embedding"]
    )
)

print(
    "Image embeddings identical:",
    torch.equal(
        phase_2_output["top_image_embeddings"].cpu(),
        loaded_output["top_image_embeddings"]
    )
)

Context embeddings identical: True
Image embeddings identical: True
